# RBC-RPMP: сравнение локального и глобального решений

Сравниваем два набора IRF, выгруженных Dynare:
- **Локальное** — `stoch_simul(order=1)` (линеаризация 1-го порядка вокруг стационара).
- **Глобальное** — `stoch_simul(order=2, pruning)` (нелинейное приближение более высокого порядка с обрезанием расходящихся членов по Kim-Kim-Schaumburg-Sims, 2008).

В литературе DSGE подход с pruning принимается как стандартный нелинейный benchmark для импульсных откликов (Schmitt-Grohé & Uribe, 2004).

Источник данных — `dynare_images_rpmp/dynare_irf_rpmp_lin.mat` и `..._glob.mat`, сгенерированы скриптом `dynare_code/run_and_save_rpmp.m`.

In [1]:
import os
from pathlib import Path
import numpy as np
from scipy.io import loadmat
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path(os.getcwd()).resolve()
IMG_DIR = REPO_ROOT / 'dynare_images_rpmp'
IMG_DIR.mkdir(exist_ok=True)

# Параметры RBC-RPMP (согласовано с RBC_RPMP.mod).
alpha   = 0.35
betta   = 0.98
delta   = 0.025
theta   = 1.25
psi     = 0.5
phi     = 16.0
nu      = 6.0
mu      = 30.0
rho_tfp = 0.2
rho_mon = 0.2
sig_tfp = 0.001
sig_mon = 0.0025
gamma   = 0.8
tau     = 1.5
piss    = 1.0
Rss     = 1.0 / betta

# --- Аналитический стационар (точно как в steady_state_model в .mod).
Rb_ss   = 1.0 / betta
mon_ss  = 1.0
tfp_ss  = 1.0
MC_ss   = 1.0
pi_ss   = 1.0
P_ss    = MC_ss * nu / (nu - 1.0)
R_ss    = P_ss * (1.0 / betta + delta - 1.0)
W_ss    = (MC_ss * ((1 - alpha) ** (1 - alpha) * alpha ** alpha) / R_ss ** alpha) ** (1.0 / (1 - alpha))
K2L     = (W_ss / R_ss) * alpha / (1.0 - alpha)
C2L     = K2L ** alpha - delta * K2L
L_ss    = ((W_ss / P_ss / phi) / (C2L ** theta)) ** (1.0 / (psi + theta))
K_ss    = K2L * L_ss
C_ss    = C2L * L_ss
Y_ss    = K_ss ** alpha * L_ss ** (1.0 - alpha)
I_ss    = delta * K_ss
Pr_ss   = (P_ss - MC_ss) * Y_ss
Wreal_ss  = W_ss / P_ss
Rreal_ss  = R_ss / P_ss
Rbreal_ss = Rb_ss / P_ss
Prreal_ss = Pr_ss / P_ss
MCreal_ss = MC_ss / P_ss

SS = {
    'Y': Y_ss, 'K': K_ss, 'L': L_ss, 'C': C_ss, 'I': I_ss,
    'W': W_ss, 'Wreal': Wreal_ss,
    'R': R_ss, 'Rreal': Rreal_ss,
    'Rb': Rb_ss, 'Rbreal': Rbreal_ss,
    'MC': MC_ss, 'MCreal': MCreal_ss,
    'Pr': Pr_ss, 'Prreal': Prreal_ss,
    'P': P_ss, 'pi': pi_ss,
    'tfp': tfp_ss, 'mon': mon_ss,
}
for k, v in SS.items():
    print(f"  {k:>6s}_ss = {v:.6f}")


       Y_ss = 0.386267
       K_ss = 2.481081
       L_ss = 0.141888
       C_ss = 0.324240
       I_ss = 0.062027
       W_ss = 1.769527
   Wreal_ss = 1.474606
       R_ss = 0.054490
   Rreal_ss = 0.045408
      Rb_ss = 1.020408
  Rbreal_ss = 0.850340
      MC_ss = 1.000000
  MCreal_ss = 0.833333
      Pr_ss = 0.077253
  Prreal_ss = 0.064378
       P_ss = 1.200000
      pi_ss = 1.000000
     tfp_ss = 1.000000
     mon_ss = 1.000000


## Загрузка IRF из Dynare

In [2]:
def load_irfs(mat_path):
    m = loadmat(str(mat_path))
    out = {}
    for k, v in m.items():
        if k.startswith('__'):
            continue
        if '_irf_' in k:
            var, shock = k.split('_irf_', 1)
            out.setdefault(shock, {})[var] = np.asarray(v).flatten()
    return out

irfs_lin  = load_irfs(IMG_DIR / 'dynare_irf_rpmp_lin.mat')
irfs_glob = load_irfs(IMG_DIR / 'dynare_irf_rpmp_glob.mat')

print("Shocks:", sorted(irfs_lin))
print("Vars  :", sorted(irfs_lin['tfp_shock']))
T = len(next(iter(irfs_lin['tfp_shock'].values())))
print(f"T = {T} периодов")


Shocks: ['mon_shock', 'tfp_shock']
Vars  : ['C', 'I', 'K', 'L', 'MC', 'MCreal', 'P', 'Pr', 'Prreal', 'R', 'Rb', 'Rbreal', 'Rreal', 'W', 'Wreal', 'Y', 'mon', 'pi', 'tfp']
T = 40 периодов


## Функция сравнения локального и глобального решений

In [3]:
def plot_compare_orders_panel(series, irfs_lin, irfs_glob, shock, ss,
                              n_rows, n_cols, title, T=None,
                              save_path=None, width=1100, height=None,
                              vspacing=0.10):
    if T is None:
        T = len(irfs_lin[shock][series[0][0]])
    t = np.arange(T)
    if height is None:
        height = 350 * n_rows + 80

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[s[2] for s in series],
        horizontal_spacing=0.10, vertical_spacing=vspacing,
    )

    show_legend = True
    for idx, (key, as_level, _title) in enumerate(series):
        r, c = idx // n_cols + 1, idx % n_cols + 1
        lin = irfs_lin[shock][key][:T]
        glb = irfs_glob[shock][key][:T]
        ss_val = ss[key]
        if as_level or abs(ss_val) < 1e-12:
            y_lin = lin
            y_glb = glb
            ylab = 'level отклонения'
        else:
            y_lin = 100.0 * lin / ss_val
            y_glb = 100.0 * glb / ss_val
            ylab = '% от SS'

        fig.add_trace(go.Scatter(
            x=t, y=y_lin, mode='lines', name='Локальное (order=1)',
            legendgroup='lin',
            line=dict(color='crimson', width=2.2, dash='dash'),
            showlegend=show_legend,
            hovertemplate='t=%{x}<br>%{y:.4f}<extra>order=1</extra>',
        ), row=r, col=c)
        fig.add_trace(go.Scatter(
            x=t, y=y_glb, mode='lines', name='Глобальное (order=2 + pruning)',
            legendgroup='glob',
            line=dict(color='royalblue', width=2.0),
            showlegend=show_legend,
            hovertemplate='t=%{x}<br>%{y:.4f}<extra>order=2</extra>',
        ), row=r, col=c)
        fig.add_hline(y=0.0, line=dict(color='gray', width=1.2, dash='dot'),
                      row=r, col=c)
        fig.update_xaxes(title_text='Периоды', row=r, col=c,
                         showgrid=True, gridcolor='lightgray')
        fig.update_yaxes(title_text=ylab, row=r, col=c,
                         showgrid=True, gridcolor='lightgray')
        show_legend = False

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center', font=dict(size=18)),
        width=width, height=height,
        legend=dict(orientation='h', yanchor='bottom', y=-0.05,
                    xanchor='center', x=0.5),
        plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(l=70, r=30, t=80, b=80),
    )

    if save_path is not None:
        save_path = Path(save_path)
        html_path = save_path.with_suffix('.html')
        fig.write_html(str(html_path), include_plotlyjs='cdn')
        print(f"Saved {html_path}")
        try:
            fig.write_image(str(save_path.with_suffix('.png')), scale=2)
            print(f"Saved {save_path.with_suffix('.png')}")
        except Exception as exc:
            print(f"PNG skipped: {type(exc).__name__}: {exc}")
    fig.show()
    return fig


def plot_compare_orders(irfs_lin, irfs_glob, shock, ss, shock_label,
                        T=None, save_path_template=None):
    series_part1 = [
        ('Y',     False, 'Выпуск Y'),
        ('K',     False, 'Капитал K'),
        ('L',     False, 'Труд L'),
        ('C',     False, 'Потребление C'),
        ('I',     False, 'Инвестиции I'),
        ('W',     False, 'Зарплата W'),
        ('Wreal', False, 'Реальная зарплата Wreal'),
        ('R',     False, 'Аренда капитала R'),
    ]
    series_part2 = [
        ('Rreal',  False, 'Реальная аренда Rreal'),
        ('Rb',     False, 'Номинальная ставка Rb'),
        ('Rbreal', False, 'Реальная Rbreal'),
        ('MC',     False, 'Предельные изд. MC'),
        ('MCreal', False, 'Реальные MCreal'),
        ('Pr',     False, 'Прибыль Pr'),
        ('Prreal', False, 'Реальная Prreal'),
        ('P',      False, 'Уровень цен P'),
        ('pi',     False, 'Инфляция π'),
        ('tfp',    False, 'TFP'),
        ('mon',    False, 'Монетарный множитель mon'),
    ]

    paths = (None, None)
    if save_path_template is not None:
        base = Path(save_path_template)
        paths = (
            base.with_name(base.stem + '_part1' + base.suffix),
            base.with_name(base.stem + '_part2' + base.suffix),
        )

    fig1 = plot_compare_orders_panel(
        series_part1, irfs_lin, irfs_glob, shock, ss,
        n_rows=4, n_cols=2,
        title=f'{shock_label}: реальные величины',
        T=T, save_path=paths[0],
    )
    fig2 = plot_compare_orders_panel(
        series_part2, irfs_lin, irfs_glob, shock, ss,
        n_rows=6, n_cols=2,
        title=f'{shock_label}: цены, прибыль, факторы',
        T=T, save_path=paths[1],
    )
    return fig1, fig2


## Монетарный шок: сравнение локального и глобального решений

Это основной результат: при шоке `mon_shock` нелинейность модели даёт заметные поправки к линейному IRF, особенно по `Rb`, `Rbreal`, `P`, `Pr`.

In [4]:
fig_mon = plot_compare_orders(
    irfs_lin, irfs_glob,
    shock='mon_shock',
    ss=SS,
    shock_label='монетарный шок',
    save_path_template=IMG_DIR / 'compare_orders_mon.png',
)


Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_mon_part1.html
Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_mon_part1.png


Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_mon_part2.html
Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_mon_part2.png


## TFP-шок: сравнение локального и глобального решений

Для контроля — те же графики, но на технологический шок. При sig_tfp = 0.001 (очень маленький шок) расхождение между order=1 и order=2 минимально.

In [5]:
fig_tfp = plot_compare_orders(
    irfs_lin, irfs_glob,
    shock='tfp_shock',
    ss=SS,
    shock_label='TFP-шок',
    save_path_template=IMG_DIR / 'compare_orders_tfp.png',
)


Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_tfp_part1.html
Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_tfp_part1.png


Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_tfp_part2.html
Saved /Users/kiri.kozlov/Desktop/diploma/dynare_images_rpmp/compare_orders_tfp_part2.png
